## ReAct/Agent + Tool 여러 개 사용 예제
 - 질문당 “최소 호출 횟수(보통 2회)”로 제한
  -  시나리오: “라이프스타일 추천 에이전트”
  - 도메인: “운동 루틴 추천”, “식단 아이디어”, “휴식/취미 추천”

3개의 Tool:
 - workout_planner : 요일/시간대에 맞는 운동 추천
 - meal_planner : 간단한 식단/간식 아이디어
 - relax_planner : 쉬는 시간, 취미, 휴식 활동 추천

에이전트 역할: 사용자의 질문을 읽고
  - 운동 같은 느낌 → workout_planner
  - 먹는 얘기/식단 → meal_planner
  - 쉬고 싶다/지친다/스트레스 → relax_planner

툴을 최대 한 번만 사용. 툴 결과 + 질문을 바탕으로 최종 한국어 답변 생성

In [15]:
!pip install -U langchain langchain-openai openai langchain-chroma langchain-community langchain-classic
!pip install sentence-transformers python-dotenv

In [22]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# MessagesPlaceholder는 LangChain 프롬프트 안에 나중에 메시지 목록이 들어갈 자리를 만들어주는 객체
# 동적인 메시지 목록을 넣기 위함
import os
from dotenv import load_dotenv
import langsmith

load_dotenv()

llm = ChatOpenAI( model="gpt-4.1-mini", temperature=0.4, )

In [23]:
# Tool 정의
@tool
def workout_planner(request: str) -> str:
    """
    가벼운 운동 루틴을 추천해 주는 도우미.
    예: '아침에 20분만 운동하고 싶어', '주 3회 운동 추천'
    """
    # 여기선 하드코딩 예시지만, 실제로는 DB, API 등 사용 가능
    base = [
        "- 5분: 가벼운 스트레칭",
        "- 10분: 스쿼트, 런지, 푸쉬업을 번갈아가며 30초씩 3세트",
        "- 5분: 마무리 스트레칭",
    ]
    plan = "\n".join(base)
    return (
        f"요청: {request}\n\n"
        "다음과 같은 간단한 루틴을 추천합니다:\n"
        f"{plan}\n\n"
        "본인 체력에 맞게 횟수와 세트 수를 조절하세요."
    )

@tool
def meal_planner(request: str) -> str:
    """
    가벼운 식단/간식 아이디어를 추천해 주는 도우미.
    예: '저녁에 너무 무겁지 않게 먹고 싶어', '공부할 때 먹을 간식 추천'
    """
    ideas = [
        "- 현미밥 + 닭가슴살 + 데친 야채",
        "- 두부 샐러드와 방울토마토",
        "- 아몬드 한 줌과 플레인 요거트",
    ]
    text = "\n".join(ideas)
    return (
        f"요청: {request}\n\n"
        "다음과 같은 가벼운 식단/간식을 고려해 볼 수 있습니다:\n"
        f"{text}\n\n"
        "물은 충분히 마시고, 너무 늦은 시간에는 과식을 피하세요."
    )

@tool
def relax_planner(request: str) -> str:
    """
    휴식, 취미, 짧은 회복 활동을 추천하는 도우미.
    예: '퇴근 후 1시간 정도 쉴만한 활동', '눈과 머리가 피곤할 때'
    """
    tips = [
        "- 10~15분 정도 가볍게 산책하기",
        "- 눈을 감고 5분간 깊게 복식호흡하기",
        "- 짧은 독서(만화, 에세이 등 가벼운 책) 20~30분",
        "- 스마트폰은 멀리 두고 스트레칭하기",
    ]
    text = "\n".join(tips)
    return (
        f"요청: {request}\n\n"
        "다음과 같은 휴식 활동을 추천합니다:\n"
        f"{text}\n\n"
        "너무 완벽하게 시간을 채우려고 하기보다, 부담 없이 할 수 있는 것을 고르는 것이 좋습니다."
    )

tools = [workout_planner, meal_planner, relax_planner]

In [24]:
# Agent용 프롬프트
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "너는 사용자의 하루 생활을 도와주는 '라이프스타일 코치' 에이전트이다.\n"
                "아래 규칙에 따라 적절한 Tool을 최대 1개까지 선택해서 사용해라.\n\n"
                "- 운동/헬스/운동 루틴/운동 계획 같은 말이 보이면 workout_planner를 사용해.\n"
                "- 식단/간식/뭐 먹지/저녁/아침/다이어트 같은 말이 보이면 meal_planner를 사용해.\n"
                "- 피곤/휴식/쉬고 싶다/스트레스/퇴근 후/집에서 할만한 것 같은 말이 보이면 relax_planner를 사용해.\n"
                "- 어느 쪽도 애매하면 Tool을 사용하지 말고, 네가 알고 있는 일반 상식으로 답해도 된다.\n\n"
                "Tool에서 반환된 텍스트는 참고자료일 뿐이다.\n"
                "최종 답변에서는:\n"
                "1) 사용자의 상황을 한두 문장으로 요약하고,\n"
                "2) 구체적인 제안 3~5개를 bullet 형태로 정리하고,\n"
                "3) 마지막에 한 문장 정도의 응원 메시지를 한국어로 덧붙여.\n"
            ),
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        # ReAct 패턴에서 Tool 호출/Observation 로그가 들어가는 자리
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [25]:
# Tool-calling Agent + Executor
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

agent_executor = AgentExecutor(  # 두뇌와 도구를 실제로 작동시키는 실행 관리자
    agent=agent,     # 판단하는 두뇌
    tools=tools,      # 사용할 수 있는 도구들
    verbose=True,    # 내부에서 어떤 Tool을 어떻게 호출하는지 보고 싶으면 True
)

In [26]:
from langchain_core.messages import HumanMessage, AIMessage
# LangChain에서 대화 기록을 저장할 때 쓰는 메시지 객체
# HumanMessage = 사용자가 한 말, AIMessage = AI가 한 말
# chat_history = 이전 대화를 순서대로 저장하는 리스트
chat_history = []

def ask_lifestyle(q: str):
    """하나의 질문에 대해 Agent를 실행하고, 히스토리를 업데이트하는 헬퍼."""
    print("\n====\n질문:", q)

    result = agent_executor.invoke(
        {
            "input": q,
            "chat_history": chat_history,
        }
    )

    print("\n최종 답변:", result["output"])

    # LangChain에서는 보통 아래처럼 메시지 객체를 쓰는 방식이 더 안전
    # 방금 한 질문과 답변을 대화 기록에 저장
    chat_history.append(HumanMessage(content=q))           # 사용자 질문 저장
    chat_history.append(AIMessage(content=result["output"]))  # AI 답변 저장
         # 왜 저장하냐? 다음 질문에서 이전 대화를 기억하게 하려고 저장

In [27]:
if __name__ == "__main__":
    q1 = "퇴근하고 집에 오면 너무 피곤한데, 30분 정도만 쉴 수 있는 활동 뭐가 좋을까?"
    q2 = "방금 추천한 활동 중에서 집 안에서 할 수 있는 것만 골라줘."
    q3 = "그 중에서 자기 전에 해도 부담 없는 루틴으로 20분짜리 계획을 만들어줘."

    ask_lifestyle(q1)
    ask_lifestyle(q2)
    ask_lifestyle(q3)


====
질문: 퇴근하고 집에 오면 너무 피곤한데, 30분 정도만 쉴 수 있는 활동 뭐가 좋을까?


> Entering new AgentExecutor chain...

Invoking: `relax_planner` with `{'request': '퇴근 후 30분 정도 쉴만한 활동 추천'}`


요청: 퇴근 후 30분 정도 쉴만한 활동 추천

다음과 같은 휴식 활동을 추천합니다:
- 10~15분 정도 가볍게 산책하기
- 눈을 감고 5분간 깊게 복식호흡하기
- 짧은 독서(만화, 에세이 등 가벼운 책) 20~30분
- 스마트폰은 멀리 두고 스트레칭하기

너무 완벽하게 시간을 채우려고 하기보다, 부담 없이 할 수 있는 것을 고르는 것이 좋습니다.퇴근 후 집에 와서 피곤할 때 30분 정도 가볍게 쉴 수 있는 활동을 찾고 계시군요.

- 10~15분 정도 가볍게 산책하며 몸과 마음을 풀어보세요.
- 눈을 감고 5분간 깊게 복식호흡을 하며 긴장을 풀어보세요.
- 만화나 에세이 같은 가벼운 책을 20~30분 정도 읽어보세요.
- 스마트폰은 잠시 멀리 두고 간단한 스트레칭을 해보세요.

부담 없이 편안하게 쉴 수 있는 활동을 선택해 보세요. 오늘 하루도 수고 많으셨어요!

> Finished chain.

최종 답변: 퇴근 후 집에 와서 피곤할 때 30분 정도 가볍게 쉴 수 있는 활동을 찾고 계시군요.

- 10~15분 정도 가볍게 산책하며 몸과 마음을 풀어보세요.
- 눈을 감고 5분간 깊게 복식호흡을 하며 긴장을 풀어보세요.
- 만화나 에세이 같은 가벼운 책을 20~30분 정도 읽어보세요.
- 스마트폰은 잠시 멀리 두고 간단한 스트레칭을 해보세요.

부담 없이 편안하게 쉴 수 있는 활동을 선택해 보세요. 오늘 하루도 수고 많으셨어요!

====
질문: 방금 추천한 활동 중에서 집 안에서 할 수 있는 것만 골라줘.


> Entering new AgentExecutor chain...

Invoking: `relax_planner` with `{'request': '퇴